# 🏙️ 城市热岛效应分析案例研究

## 📚 研究背景

**城市热岛效应**（Urban Heat Island, UHI）是指城市中心区域的温度明显高于周边郊区或乡村的现象。

**研究目标**:
1. 对比北京、上海、广州三个城市的地表温度
2. 分析植被覆盖与温度的关系
3. 评估不同城市的热岛强度

**数据来源**:
- 地表温度: MODIS/061/MOD11A2
- 植被指数: MODIS/061/MOD13Q1

**时间范围**: 2023年全年

---

## 🚀 第一步：环境设置

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
from shapely.geometry import Point
from scipy import stats
import ee

# 初始化GEE
ee.Initialize()

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

print("✅ 环境设置完成！")

## 📍 第二步：定义研究区域

In [ ]:
# 定义三个城市的边界
cities = {
    '北京': {
        'bbox': (116.0, 39.7, 116.8, 40.2),
        'center': (116.4, 39.9),
        'climate': '温带季风气候',
        'population': '2154万'
    },
    '上海': {
        'bbox': (121.0, 31.0, 121.8, 31.5),
        'center': (121.4, 31.2),
        'climate': '亚热带季风气候',
        'population': '2487万'
    },
    '广州': {
        'bbox': (113.0, 23.0, 113.5, 23.5),
        'center': (113.3, 23.1),
        'climate': '亚热带季风气候',
        'population': '1867万'
    }
}

print("🏙️ 研究城市信息:")
for city, info in cities.items():
    print(f"\n{city}:")
    print(f"  气候: {info['climate']}")
    print(f"  人口: {info['population']}")
    print(f"  中心: {info['center']}")

## 📥 第三步：数据提取函数

In [ ]:
def create_sampling_points(bbox, spacing=0.05):
    """创建采样点"""
    min_lon, min_lat, max_lon, max_lat = bbox
    lons = np.arange(min_lon, max_lon, spacing)
    lats = np.arange(min_lat, max_lat, spacing)
    
    points = []
    for lon in lons:
        for lat in lats:
            points.append({
                'geometry': Point(lon, lat),
                'longitude': lon,
                'latitude': lat
            })
    
    return gpd.GeoDataFrame(points, crs='EPSG:4326')

def extract_lst_data(points_df, start_date, end_date):
    """提取地表温度数据"""
    # 创建FeatureCollection
    features = []
    for idx, row in points_df.iterrows():
        geom = row.geometry
        point = ee.Geometry.Point([geom.x, geom.y])
        feature = ee.Feature(point, {'id': idx})
        features.append(feature)
    
    fc = ee.FeatureCollection(features)
    
    # 获取MODIS LST数据
    collection = ee.ImageCollection('MODIS/061/MOD11A2')\
        .filterDate(start_date, end_date)\
        .select('LST_Day_1km')
    
    lst_mean = collection.mean()
    results = lst_mean.sampleRegions(collection=fc, scale=1000)
    
    data = results.getInfo()['features']
    extracted_data = []
    for item in data:
        props = item['properties']
        extracted_data.append({
            'id': props.get('id'),
            'LST_raw': props.get('LST_Day_1km')
        })
    
    return pd.DataFrame(extracted_data)

def extract_ndvi_data(points_df, start_date, end_date):
    """提取NDVI数据"""
    features = []
    for idx, row in points_df.iterrows():
        geom = row.geometry
        point = ee.Geometry.Point([geom.x, geom.y])
        feature = ee.Feature(point, {'id': idx})
        features.append(feature)
    
    fc = ee.FeatureCollection(features)
    
    collection = ee.ImageCollection('MODIS/061/MOD13Q1')\
        .filterDate(start_date, end_date)\
        .select('NDVI')
    
    ndvi_mean = collection.mean()
    results = ndvi_mean.sampleRegions(collection=fc, scale=250)
    
    data = results.getInfo()['features']
    extracted_data = []
    for item in data:
        props = item['properties']
        extracted_data.append({
            'id': props.get('id'),
            'NDVI_raw': props.get('NDVI')
        })
    
    return pd.DataFrame(extracted_data)

print("✅ 数据提取函数已定义！")

## 📊 第四步：提取所有城市的数据

In [ ]:
# 存储所有城市的数据
all_data = {}

for city_name, city_info in cities.items():
    print(f"\n📍 处理 {city_name}...")
    
    # 创建采样点
    points = create_sampling_points(city_info['bbox'], spacing=0.1)
    print(f"   采样点数: {len(points)}")
    
    # 提取LST数据
    lst_data = extract_lst_data(points, '2023-01-01', '2023-12-31')
    lst_data['LST_Celsius'] = lst_data['LST_raw'] * 0.02 - 273.15
    
    # 提取NDVI数据
    ndvi_data = extract_ndvi_data(points, '2023-01-01', '2023-12-31')
    ndvi_data['NDVI_scaled'] = ndvi_data['NDVI_raw'] * 0.0001
    
    # 合并数据
    city_data = pd.merge(lst_data, ndvi_data, on='id')
    city_data['longitude'] = points.geometry.x.values
    city_data['latitude'] = points.geometry.y.values
    city_data['city'] = city_name
    
    all_data[city_name] = city_data
    
    print(f"   平均温度: {city_data['LST_Celsius'].mean():.2f}°C")
    print(f"   平均NDVI: {city_data['NDVI_scaled'].mean():.4f}")

# 合并所有城市数据
combined_data = pd.concat(all_data.values(), ignore_index=True)
print(f"\n✅ 数据提取完成！总数据点: {len(combined_data)}")

## 📈 第五步：城市温度对比分析

In [ ]:
# 计算每个城市的统计指标
city_stats = []
for city_name, data in all_data.items():
    stats = {
        '城市': city_name,
        '平均温度(°C)': data['LST_Celsius'].mean(),
        '最高温度(°C)': data['LST_Celsius'].max(),
        '最低温度(°C)': data['LST_Celsius'].min(),
        '温度标准差': data['LST_Celsius'].std(),
        '平均NDVI': data['NDVI_scaled'].mean(),
        '数据点数': len(data)
    }
    city_stats.append(stats)

stats_df = pd.DataFrame(city_stats)

# 显示统计表
print("\n📊 三城市地表温度和NDVI对比 (2023年):")
print(stats_df.to_string(index=False))

# 可视化对比
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 温度对比
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
axes[0].bar(stats_df['城市'], stats_df['平均温度(°C)'], color=colors, alpha=0.7)
axes[0].set_ylabel('平均温度 (°C)', fontsize=12)
axes[0].set_title('城市平均地表温度对比', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')

# NDVI对比
axes[1].bar(stats_df['城市'], stats_df['平均NDVI'], color=colors, alpha=0.7)
axes[1].set_ylabel('平均NDVI', fontsize=12)
axes[1].set_title('城市平均植被覆盖对比', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 🌡️ 第六步：温度-植被关系分析

In [ ]:
# 创建散点图分析温度与NDVI的关系
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (city_name, data) in enumerate(all_data.items()):
    ax = axes[i]
    
    # 散点图
    scatter = ax.scatter(
        data['NDVI_scaled'],
        data['LST_Celsius'],
        c=data['LST_Celsius'],
        cmap='RdYlBu_r',
        alpha=0.6,
        s=50,
        edgecolors='black',
        linewidths=0.5
    )
    
    # 添加趋势线
    z = np.polyfit(data['NDVI_scaled'], data['LST_Celsius'], 1)
    p = np.poly1d(z)
    x_trend = np.linspace(data['NDVI_scaled'].min(), data['NDVI_scaled'].max(), 100)
    ax.plot(x_trend, p(x_trend), "r--", alpha=0.8, linewidth=2)
    
    # 计算相关系数
    corr = data['NDVI_scaled'].corr(data['LST_Celsius'])
    
    ax.set_xlabel('NDVI', fontsize=12)
    ax.set_ylabel('温度 (°C)', fontsize=12)
    ax.set_title(f'{city_name}\n相关系数: r = {corr:.3f}', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    plt.colorbar(scatter, ax=ax, label='温度 (°C)')

plt.tight_layout()
plt.show()

# 打印相关性分析结果
print("\n📊 温度-NDVI相关性分析:")
for city_name, data in all_data.items():
    corr = data['NDVI_scaled'].corr(data['LST_Celsius'])
    
    # 相关系数显著性检验
    t_stat, p_value = stats.pearsonr(data['NDVI_scaled'], data['LST_Celsius'])
    
    print(f"\n{city_name}:")
    print(f"  相关系数: r = {corr:.4f}")
    print(f"  P值: {p_value:.4e}")
    
    if p_value < 0.001:
        sig = "*** (极显著)"
    elif p_value < 0.01:
        sig = "** (很显著)"
    elif p_value < 0.05:
        sig = "* (显著)"
    else:
        sig = "(不显著)"
    
    print(f"  显著性: {sig}")
    
    if corr < -0.3:
        print(f"  解释: NDVI与温度呈显著负相关，植被覆盖有助于降低温度")
    elif corr > 0.3:
        print(f"  解释: NDVI与温度呈正相关，可能与其他因素相关")
    else:
        print(f"  解释: NDVI与温度相关性较弱")

## 🏆 第七步：热岛强度评估

In [ ]:
# 计算每个城市的热岛强度（温度标准差作为指标）
print("\n🌡️ 城市热岛强度评估:")
print("="*60)

uhi_results = []
for city_name, data in all_data.items():
    # 使用温度标准差作为热岛强度指标
    # 标准差越大，表示城市内部温度差异越大，热岛效应越明显
    uhi_intensity = data['LST_Celsius'].std()
    
    # 计算温度范围
    temp_range = data['LST_Celsius'].max() - data['LST_Celsius'].min()
    
    # 根据标准差分类
    if uhi_intensity > 3:
        level = "强热岛"
    elif uhi_intensity > 2:
        level = "中等热岛"
    elif uhi_intensity > 1:
        level = "弱热岛"
    else:
        level = "无明显热岛"
    
    result = {
        '城市': city_name,
        '热岛强度': uhi_intensity,
        '温度范围': temp_range,
        '热岛等级': level
    }
    uhi_results.append(result)
    
    print(f"\n{city_name}:")
    print(f"  热岛强度(σ): {uhi_intensity:.2f}°C")
    print(f"  温度范围: {temp_range:.2f}°C")
    print(f"  热岛等级: {level}")

# 可视化热岛强度
uhi_df = pd.DataFrame(uhi_results)

fig, ax = plt.subplots(figsize=(10, 6))
colors_map = {'强热岛': '#FF6B6B', '中等热岛': '#FFA07A', '弱热岛': '#98D8C8', '无明显热岛': '#45B7D1'}

for level in uhi_df['热岛等级'].unique():
    subset = uhi_df[uhi_df['热岛等级'] == level]
    ax.bar(subset['城市'], subset['热岛强度'], 
           color=colors_map[level], label=level, alpha=0.7)

ax.set_ylabel('热岛强度 (°C)', fontsize=12)
ax.set_title('城市热岛强度对比', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 📝 第八步：研究结论

In [ ]:
print("\n" + "="*60)
print("📊 城市热岛效应研究结论")
print("="*60)

print("\n1️⃣ 温度对比:")
hottest = stats_df.loc[stats_df['平均温度(°C)'].idxmax()]
coldest = stats_df.loc[stats_df['平均温度(°C)'].idxmin()]
print(f"   最热城市: {hottest['城市']} ({hottest['平均温度(°C)']:.2f}°C)")
print(f"   最冷城市: {coldest['城市']} ({coldest['平均温度(°C)']:.2f}°C)")
print(f"   温差: {hottest['平均温度(°C)'] - coldest['平均温度(°C)']:.2f}°C")

print("\n2️⃣ 植被覆盖对比:")
greenest = stats_df.loc[stats_df['平均NDVI'].idxmax()]
least_green = stats_df.loc[stats_df['平均NDVI'].idxmin()]
print(f"   植被最好: {greenest['城市']} (NDVI={greenest['平均NDVI']:.4f})")
print(f"   植被最少: {least_green['城市']} (NDVI={least_green['平均NDVI']:.4f})")

print("\n3️⃣ 热岛强度:")
strongest_uhi = uhi_df.loc[uhi_df['热岛强度'].idxmax()]
print(f"   热岛最强: {strongest_uhi['城市']} ({strongest_uhi['热岛等级']})")

print("\n4️⃣ 温度-植被关系:")
print("   总体趋势: 植被覆盖与温度呈负相关")
print("   城市绿化有助于缓解热岛效应")

print("\n5️⃣ 建议:")
print("   • 增加城市绿地面积
      "   • 建设城市公园和绿化带
      "   • 推广屋顶绿化
      "   • 保护现有植被")

print("\n" + "="*60)
print("✅ 研究完成！")
print("="*60)

---

## 🎯 案例研究总结

### 📚 研究方法

1. **数据提取**: 使用GEE提取2023年MODIS LST和NDVI数据
2. **数据处理**: 应用缩放因子转换物理单位
3. **统计分析**: 计算温度和NDVI的统计指标
4. **相关性分析**: 分析温度与植被的关系
5. **可视化**: 创建多种图表展示结果

### 🔬 主要发现

- 不同城市存在明显的温度差异
- 植被覆盖与温度呈负相关
- 城市热岛强度各不相同
- 城市绿化是缓解热岛效应的有效手段

### 💡 研究启示

这个案例展示了如何使用LST数据提取工具进行实际研究。你可以：

1. **修改城市**: 研究你感兴趣的城市
2. **改变时间范围**: 分析不同季节或年份
3. **添加变量**: 引入更多环境因子
4. **改进方法**: 尝试不同的分析方法

### 📖 相关文献

- Oke, T. R. (1982). The energetic basis of the urban heat island.
- 严平, 等. (2018). 中国城市热岛效应研究进展.
- 徐祥德, 等. (2020). 城市植被与热环境关系.

---

**恭喜！你已完成城市热岛效应分析案例研究！** 🎉

继续探索:
- 🚀 尝试更多案例: `QUICK_TUTORIAL.ipynb`
- 📚 查看完整文档: `COMPLETE_GUIDE.md`
- 🔧 探索高级功能: `LST_Tools_Master.ipynb`